# Multi-Seed Robustness Runs — Pedestrian Crossing-Intention BiLSTM

**Purpose.** The thesis results so far are from a single training seed (42).
This notebook re-trains each model over **5 seeds** and reports **mean ± std**
for every test metric, so the headline numbers are statistically grounded
(e.g. `AUC 0.930 ± 0.003`) instead of a single run.

**What it trains (each over 5 seeds):**
1. `bilstm_baseline` — 5-D (bbox + ego-speed), the main model.
2. `bilstm_bbox_only` — 4-D (bbox only) ablation.
3. `bilstm_attention` — 5-D + temporal attention.

**Contract preserved exactly** (must match `04_train_bilstm.py`): feature order
`[x1,y1,x2,y2,vehicle_speed]` raw PIE pixels, train-only `(x-mean)/std`,
`obs_len=16`, `POS_WEIGHT=1.44` fixed, split by recording set
(train=set01/02/04, val=set05/06, **test=set03**), early stop on val AUC
(patience 15), test touched once on the best-val checkpoint, threshold 0.5.
Seed 42 reproduces the existing Day-5 numbers as a sanity check.

**Inputs required** (attach as a Kaggle Dataset, see the instructions I gave
you in chat): `X.npy`, `y.npy`, `meta.pkl` (your `sequences/` folder).

**Outputs** (in `/kaggle/working/`): per-seed `final.json`, a full results CSV,
and a `multiseed_summary.md` table you can paste straight into the thesis.


In [ ]:
# === Cell 1: imports + environment check ===
import json, pickle, random, time, math
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    precision_score, recall_score, confusion_matrix,
)

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

In [ ]:
# === Cell 2: CONFIG — edit here ===

# Seeds to run. 42 first so it reproduces your existing single-seed numbers.
SEEDS = [42, 0, 1, 7, 123]

# Which models to run. Comment out any you don't want.
RUN_MODELS = [
    "bilstm_baseline",   # 5-D, the main model
    "bilstm_bbox_only",  # 4-D ablation (drops ego-speed)
    "bilstm_attention",  # 5-D + temporal attention
]

# Fixed across ALL runs (do not change — these define the locked contract).
POS_WEIGHT  = 1.44          # 819 neg / 570 pos
TRAIN_SETS  = {"set01", "set02", "set04"}
VAL_SETS    = {"set05", "set06"}
TEST_SETS   = {"set03"}
EPOCHS      = 100
BATCH_SIZE  = 32
LR          = 1e-3
WEIGHT_DECAY= 1e-5
PATIENCE    = 15
THRESHOLD   = 0.5

OUT_ROOT = Path("/kaggle/working/runs_multiseed")
OUT_ROOT.mkdir(parents=True, exist_ok=True)


def find_seq_dir():
    """Locate the folder containing X.npy/y.npy/meta.pkl under /kaggle/input."""
    base = Path("/kaggle/input")
    hits = list(base.rglob("X.npy"))
    if not hits:
        raise FileNotFoundError(
            "X.npy not found under /kaggle/input. Attach your sequences dataset "
            "(X.npy, y.npy, meta.pkl) via 'Add Input' on the right panel."
        )
    seq_dir = hits[0].parent
    for f in ("X.npy", "y.npy", "meta.pkl"):
        if not (seq_dir / f).exists():
            raise FileNotFoundError(f"{f} missing in {seq_dir}")
    return seq_dir

SEQ_DIR = find_seq_dir()
print("sequences dir:", SEQ_DIR)


In [ ]:
# === Cell 3: model definitions (copied verbatim from 03 / 03b / 07) ===

class BiLSTMIntentPredictor(nn.Module):
    """Baseline: input proj -> 2-layer BiLSTM -> last timestep -> head."""
    def __init__(self, input_dim=5, proj_dim=64, hidden_dim=128,
                 num_layers=2, dropout=0.3):
        super().__init__()
        self.input_proj = nn.Sequential(nn.Linear(input_dim, proj_dim), nn.ReLU())
        self.bilstm = nn.LSTM(proj_dim, hidden_dim, num_layers,
                              dropout=dropout, bidirectional=True, batch_first=True)
        self.head = nn.Linear(hidden_dim * 2, 1)
    def forward(self, x):
        out, _ = self.bilstm(self.input_proj(x))
        return self.head(out[:, -1, :])


class BiLSTMIntentPredictorFlex(nn.Module):
    """Same arch, configurable input_dim. input_dim=4 = bbox-only."""
    def __init__(self, input_dim=5, hidden_size=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, 64)
        self.bilstm = nn.LSTM(64, hidden_size, num_layers,
                              dropout=dropout if num_layers > 1 else 0.0,
                              bidirectional=True, batch_first=True)
        self.head = nn.Linear(hidden_size * 2, 1)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        x = self.drop(torch.relu(self.input_proj(x)))
        out, _ = self.bilstm(x)
        return self.head(self.drop(out[:, -1, :]))


class BiLSTMAttentionIntentPredictor(nn.Module):
    """Baseline backbone + additive temporal attention over all T timesteps."""
    def __init__(self, input_dim=5, hidden_size=128, num_layers=2,
                 dropout=0.3, attn_dim=64):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, 64)
        self.bilstm = nn.LSTM(64, hidden_size, num_layers,
                              dropout=dropout if num_layers > 1 else 0.0,
                              bidirectional=True, batch_first=True)
        self.attn_W = nn.Linear(hidden_size * 2, attn_dim)
        self.attn_v = nn.Linear(attn_dim, 1, bias=False)
        self.head = nn.Linear(hidden_size * 2, 1)
        self.drop = nn.Dropout(dropout)
    def forward(self, x, return_attn=False):
        x = self.drop(torch.relu(self.input_proj(x)))
        H, _ = self.bilstm(x)
        scores = self.attn_v(torch.tanh(self.attn_W(H)))
        weights = torch.softmax(scores, dim=1)
        context = (weights * H).sum(dim=1)
        logit = self.head(self.drop(context))
        if return_attn:
            return logit, weights.squeeze(-1)
        return logit


def build_model(name):
    if name == "bilstm_baseline":
        return BiLSTMIntentPredictor(input_dim=5)
    if name == "bilstm_bbox_only":
        return BiLSTMIntentPredictorFlex(input_dim=4)
    if name == "bilstm_attention":
        return BiLSTMAttentionIntentPredictor(input_dim=5)
    raise ValueError(name)

# Which models drop the ego-speed column (index 4)?
USE_SPEED = {"bilstm_baseline": True, "bilstm_bbox_only": False, "bilstm_attention": True}


In [ ]:
# === Cell 4: data utils (split / normalize / evaluate / seed) ===

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def load_raw(seq_dir):
    X = np.load(seq_dir / "X.npy").astype(np.float32)
    y = np.load(seq_dir / "y.npy").astype(np.float32)
    with open(seq_dir / "meta.pkl", "rb") as f:
        meta = pickle.load(f)
    # meta may be a list of dicts OR a DataFrame — handle both.
    if isinstance(meta, pd.DataFrame):
        set_ids = meta["set_id"].to_numpy()
    else:
        set_ids = np.array([m["set_id"] for m in meta])
    return X, y, set_ids


def split(X, y, set_ids):
    tr = np.isin(set_ids, list(TRAIN_SETS))
    va = np.isin(set_ids, list(VAL_SETS))
    te = np.isin(set_ids, list(TEST_SETS))
    return X[tr], y[tr], X[va], y[va], X[te], y[te]


def norm_stats(Xtr):
    flat = Xtr.reshape(-1, Xtr.shape[-1])
    return flat.mean(axis=0), flat.std(axis=0) + 1e-6


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    crit = nn.BCEWithLogitsLoss(reduction="sum")
    probs_all, labels_all, total, n = [], [], 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        logits = model(xb).squeeze(-1)
        total += crit(logits, yb).item()
        probs_all.append(torch.sigmoid(logits).cpu().numpy())
        labels_all.append(yb.cpu().numpy())
        n += yb.size(0)
    probs = np.concatenate(probs_all); labels = np.concatenate(labels_all)
    preds = (probs >= THRESHOLD).astype(int)
    return {
        "loss": total / n,
        "acc":  accuracy_score(labels, preds),
        "f1":   f1_score(labels, preds, zero_division=0),
        "auc":  roc_auc_score(labels, probs) if len(np.unique(labels)) > 1 else float("nan"),
        "prec": precision_score(labels, preds, zero_division=0),
        "rec":  recall_score(labels, preds, zero_division=0),
        "probs": probs, "labels": labels, "preds": preds,
    }

# Load once; X is sliced per-model inside the training loop.
X_ALL, Y_ALL, SETIDS_ALL = load_raw(SEQ_DIR)
print("X:", X_ALL.shape, "| y pos rate:", round(float(Y_ALL.mean()), 3))

In [ ]:
# === Cell 5: train one (model, seed) -> final test metrics ===

def make_loader(X, y, shuffle):
    ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=2, pin_memory=(DEVICE.type == "cuda"))


def train_one(model_name, seed, verbose=False):
    set_seed(seed)
    out_dir = OUT_ROOT / f"{model_name}_seed{seed}"
    out_dir.mkdir(parents=True, exist_ok=True)

    # Slice features: bbox-only drops the ego-speed column (index 4).
    X = X_ALL if USE_SPEED[model_name] else X_ALL[:, :, :4]
    Xtr, ytr, Xva, yva, Xte, yte = split(X, Y_ALL, SETIDS_ALL)

    mean, std = norm_stats(Xtr)
    Xtr, Xva, Xte = (Xtr - mean) / std, (Xva - mean) / std, (Xte - mean) / std
    np.save(out_dir / "norm_mean.npy", mean)
    np.save(out_dir / "norm_std.npy", std)

    train_loader = make_loader(Xtr, ytr, True)
    val_loader   = make_loader(Xva, yva, False)
    test_loader  = make_loader(Xte, yte, False)

    model = build_model(model_name).to(DEVICE)
    crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT], device=DEVICE))
    opt  = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max",
                                                       factor=0.5, patience=5)

    best_auc, no_improve = -1.0, 0
    for epoch in range(1, EPOCHS + 1):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(xb).squeeze(-1), yb)
            loss.backward(); opt.step()
        val = evaluate(model, val_loader)
        sched.step(val["auc"])
        if verbose:
            print(f"  ep {epoch:3d} vAUC {val['auc']:.3f} vF1 {val['f1']:.3f}")
        if val["auc"] > best_auc:
            best_auc, no_improve = val["auc"], 0
            torch.save({"model": model.state_dict(), "epoch": epoch,
                        "val_metrics": {k: v for k, v in val.items()
                                        if k not in ("probs", "labels", "preds")}},
                       out_dir / "best.pt")
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                break

    # final test on best-val checkpoint (touched once)
    ckpt = torch.load(out_dir / "best.pt", map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt["model"])
    test = evaluate(model, test_loader)
    cm = confusion_matrix(test["labels"], test["preds"]).tolist()
    final = {
        "model": model_name, "seed": seed, "best_epoch": ckpt["epoch"],
        "test": {k: float(v) for k, v in test.items()
                 if k not in ("probs", "labels", "preds")},
        "test_confusion_matrix": cm,
    }
    with open(out_dir / "final.json", "w") as f:
        json.dump(final, f, indent=2)
    return final


In [ ]:
# === Cell 6: run all models x all seeds ===
rows = []
t_start = time.time()
for model_name in RUN_MODELS:
    for seed in SEEDS:
        t0 = time.time()
        res = train_one(model_name, seed)
        t = res["test"]
        rows.append({"model": model_name, "seed": seed,
                     "best_epoch": res["best_epoch"],
                     "auc": t["auc"], "f1": t["f1"], "acc": t["acc"],
                     "prec": t["prec"], "rec": t["rec"]})
        print(f"{model_name:18s} seed {seed:3d} | "
              f"AUC {t['auc']:.3f} F1 {t['f1']:.3f} Acc {t['acc']:.3f} "
              f"P {t['prec']:.3f} R {t['rec']:.3f} | "
              f"ep {res['best_epoch']:2d} | {time.time()-t0:.0f}s")
print(f"\nTOTAL: {time.time()-t_start:.0f}s")

results = pd.DataFrame(rows)
results.to_csv("/kaggle/working/multiseed_results.csv", index=False)
results


In [ ]:
# === Cell 7: aggregate -> mean +/- std per model ===
METRICS = ["auc", "f1", "acc", "prec", "rec"]
agg = results.groupby("model")[METRICS].agg(["mean", "std"])

summary = pd.DataFrame(index=results["model"].unique())
for m in METRICS:
    mean = agg[(m, "mean")]
    std  = agg[(m, "std")]
    summary[m] = [f"{mu:.3f} +/- {sd:.3f}" for mu, sd in zip(mean, std)]
summary.index.name = "model"
summary = summary.reindex(RUN_MODELS)
summary.to_csv("/kaggle/working/multiseed_summary.csv")
print("Mean +/- std over", len(SEEDS), "seeds:", SEEDS)
summary


In [ ]:
# === Cell 8: write a markdown table for the thesis ===
lines = [f"# Multi-seed results (mean ± std over {len(SEEDS)} seeds: {SEEDS})", "",
         "Test set = PIE set03. Contract identical to single-seed runs "
         "(POS_WEIGHT=1.44, obs_len=16, early stop on val AUC, threshold 0.5).", "",
         "| Model | AUC | F1 | Accuracy | Precision | Recall |",
         "|---|---|---|---|---|---|"]
name_map = {"bilstm_baseline": "BiLSTM 5-D (baseline)",
            "bilstm_bbox_only": "BiLSTM 4-D (bbox-only)",
            "bilstm_attention": "BiLSTM 5-D + attention"}
for model_name in RUN_MODELS:
    sub = results[results["model"] == model_name]
    cells_md = []
    for m in METRICS:
        cells_md.append(f"{sub[m].mean():.3f} ± {sub[m].std():.3f}")
    lines.append(f"| {name_map.get(model_name, model_name)} | " + " | ".join(cells_md) + " |")
lines += ["", "Per-seed detail:", "",
          "| Model | Seed | AUC | F1 | Acc | P | R | best epoch |",
          "|---|---|---|---|---|---|---|---|"]
for _, r in results.iterrows():
    lines.append(f"| {r['model']} | {int(r['seed'])} | {r['auc']:.3f} | "
                 f"{r['f1']:.3f} | {r['acc']:.3f} | {r['prec']:.3f} | "
                 f"{r['rec']:.3f} | {int(r['best_epoch'])} |")
md_text = "\n".join(lines) + "\n"
with open("/kaggle/working/multiseed_summary.md", "w") as f:
    f.write(md_text)
print(md_text)


## After it finishes

1. Open the **Output** tab (right panel) — you'll find:
   - `multiseed_summary.md` — the paste-ready table (mean ± std) for your thesis.
   - `multiseed_summary.csv` and `multiseed_results.csv` — the same numbers as data.
   - `runs_multiseed/<model>_seed<N>/` — each run's `best.pt`, `final.json`,
     `norm_mean.npy`, `norm_std.npy` (kept in case you want any checkpoint).
2. **Sanity check:** the `seed 42` row for `bilstm_baseline` should reproduce your
   existing single-seed numbers (AUC ≈ 0.931). If it does, the multi-seed numbers
   are trustworthy.
3. Download the whole output with the **Download** button, or click individual files.

**Expected runtime:** ~1 min per run on the T4 GPU → 3 models × 5 seeds ≈ **15 min**.
If you only want the baseline first, set `RUN_MODELS = ["bilstm_baseline"]` in Cell 2.
